# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Copy the `trycloudflare.com` URL into your local CircuitAI frontend.

In [ ]:
# ── Cell 1: Install system dependencies ────────────────────────
!apt-get install -y poppler-utils -q

# Install Cloudflared for the public tunnel
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('System deps installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Pre-load model (keeps subsequent requests fast) ────
from model_loader import load_model
load_model()
print('Model ready ✓')

In [ ]:
# ── Cell 4: Start FastAPI server + Cloudflare Tunnel ───────────
import subprocess, threading, time
from main import start_server

# Start the API server in a background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)  # Give uvicorn time to bind

# Start Cloudflare tunnel and capture the public URL
print('Starting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print(f'\n' + '='*60)
            print(f'  🔥 PUBLIC URL: {public_url}')
            print(f'  Paste this into CircuitAI frontend > Connection field')
            print('='*60)
        break